In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import faiss
from dotenv import load_dotenv
import os
from google import genai

---

In [2]:
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
EMBED_MODEL = os.getenv("EMBED_MODEL")
GEN_MODEL = os.getenv("GEN_MODEL")
OUTPUT_DIMENSION = os.getenv("OUTPUT_DIMENSION")

client = genai.Client(api_key=GEMINI_API_KEY)

In [3]:
index = faiss.read_index("../output_index/text-embed-04")

---

# Process

In [14]:
movie_df = pd.read_csv("../data/movie_infos.csv")
movie_df["vectorID"] = movie_df.index.values
movie_df

,movieId,title,genres,tags,year,rating,imdbId,vectorID
0,1,Toy Story (1995),"Adventure,Animation,Children,Comedy,Fantasy","adventure,animated,animation,cartoon,cgi,child...",1995.0,3.921240,tt0114709,0
1,2,Jumanji (1995),"Adventure,Children,Fantasy","adventure,animals,big budget,childhood,childre...",1995.0,3.211977,tt0113497,1
2,3,Grumpier Old Men (1995),"Comedy,Romance","comedy,good sequel,original,sequel,sequels",1995.0,3.151040,tt0113228,2
3,4,Waiting to Exhale (1995),"Comedy,Drama,Romance","chick flick,girlie movie,romantic,unlikely fri...",1995.0,2.861393,tt0114885,3
4,5,Father of the Bride Part II (1995),Comedy,"comedy,destiny,family,father daughter relation...",1995.0,3.064592,tt0113041,4
...,...,...,...,...,...,...,...,...
10337,130578,The Gunman (2015),"Action,Thriller","action,assassin,assassination,good action,real...",2015.0,3.000000,tt2515034,10337
10338,130840,Spring (2015),"Horror,Romance,Sci-Fi","cinematography,creepy,horror,immortality,love ...",2015.0,3.500000,tt3395184,10338
10339,131013,Get Hard (2015),Comedy,"buddy movie,coen bros,comedy,crude humor,foul ...",2015.0,2.500000,tt2561572,10339
10340,131168,Phoenix (2014),Drama,"betrayal,camp,cinematography,criterion,dramati...",2014.0,3.500000,tt2764784,10340


In [10]:
ratings = pd.read_csv("../data/rating.csv")

In [ ]:
test = movie_df.merge(ratings.groupby(by="movieId").count()
                      ["userId"], on="movieId", how="left")
test.fillna(0)

C = test["rating"].mean()
m = 1300


def weight_rating(row):

    v = row["userId"]
    R = row["rating"]

    return (v/(v+m)) * R + (m/(v+m)) * C


movie_df["weight_rating"] = test.apply(weight_rating, axis=1)


def convert_prompt(row):
    title = row["title"]
    genres = row["genres"]
    tags = row["tags"]
    # link = f"https://www.imdb.com/title/{row["imdbId"]}/"
    # rating = round(row["weight_rating"],4)

    return f"Movie's title: {title}\ngenres: {genres}\ntags: {tags}"


movie_df["page_content"] = movie_df.apply(convert_prompt, axis=1)
movie_df.set_index("movieId", inplace=True)

In [ ]:
movie_df = movie_df[["vectorID", "title", "genres", "tags",
                     "weight_rating", "imdbId", "page_content"]]

In [ ]:
# movie_df.to_csv("../data/movie_infos_2.csv")

---

# Data

In [4]:
movie_df = pd.read_csv("../data/movie_infos_2.csv")
movie_df.set_index("movieId", inplace=True)

In [5]:
movie_df

,vectorID,title,genres,tags,weight_rating,imdbId,page_content
movieId,,,,,,,
1,0,Toy Story (1995),"Adventure,Animation,Children,Comedy,Fantasy","adventure,animated,animation,cartoon,cgi,child...",3.904516,tt0114709,Movie's title: Toy Story (1995)\ngenres: Adven...
2,1,Jumanji (1995),"Adventure,Children,Fantasy","adventure,animals,big budget,childhood,childre...",3.214918,tt0113497,Movie's title: Jumanji (1995)\ngenres: Adventu...
3,2,Grumpier Old Men (1995),"Comedy,Romance","comedy,good sequel,original,sequel,sequels",3.161619,tt0113228,Movie's title: Grumpier Old Men (1995)\ngenres...
4,3,Waiting to Exhale (1995),"Comedy,Drama,Romance","chick flick,girlie movie,romantic,unlikely fri...",2.990833,tt0114885,Movie's title: Waiting to Exhale (1995)\ngenre...
5,4,Father of the Bride Part II (1995),Comedy,"comedy,destiny,family,father daughter relation...",3.083970,tt0113041,Movie's title: Father of the Bride Part II (19...
...,...,...,...,...,...,...,...
130578,10337,The Gunman (2015),"Action,Thriller","action,assassin,assassination,good action,real...",3.264228,tt2515034,Movie's title: The Gunman (2015)\ngenres: Acti...
130840,10338,Spring (2015),"Horror,Romance,Sci-Fi","cinematography,creepy,horror,immortality,love ...",3.265785,tt3395184,"Movie's title: Spring (2015)\ngenres: Horror,R..."
131013,10339,Get Hard (2015),Comedy,"buddy movie,coen bros,comedy,crude humor,foul ...",3.262312,tt2561572,Movie's title: Get Hard (2015)\ngenres: Comedy...


In [6]:
user_rates = pd.read_pickle("../data/sorted_ratings.pkl")
user_rates

rating           timestamp
userId movieId                            
1      924         3.5 2004-09-10 03:06:38
       919         3.5 2004-09-10 03:07:01
       2683        3.5 2004-09-10 03:07:30
       1584        3.5 2004-09-10 03:07:36
       1079        4.0 2004-09-10 03:07:45
...                ...                 ...
138493 6534        3.0 2009-12-07 18:18:28
       53464       4.0 2009-12-07 18:18:40
       1275        3.0 2010-01-01 20:42:32
       6996        3.0 2010-01-01 20:42:35
       405         3.0 2010-01-01 20:42:52

[19792548 rows x 2 columns]

---

# Luồng chạy

### Lấy ra tất cả các phim user bất kì đã xem

In [7]:
movie_list = user_rates.loc[1].merge(movie_df, on="movieId")[
    ["vectorID", "title", "genres", "tags", "rating", "weight_rating", "timestamp", "page_content"]]

In [8]:
movie_list

,vectorID,title,genres,tags,rating,weight_rating,timestamp,page_content
movieId,,,,,,,,
924,817,2001: A Space Odyssey (1968),"Adventure,Drama,Sci-Fi","70mm,adapted from:book,afi 100,allegory,amazin...",3.5,3.921942,2004-09-10 03:06:38,Movie's title: 2001: A Space Odyssey (1968)\ng...
919,812,"Wizard of Oz, The (1939)","Adventure,Children,Fantasy,Musical","adapted from:book,afi 100 (movie quotes),based...",3.5,3.944042,2004-09-10 03:07:01,"Movie's title: Wizard of Oz, The (1939)\ngenre..."
2683,2351,Austin Powers: The Spy Who Shagged Me (1999),"Action,Adventure,Comedy","comedy,crude humor,dumb,dumb but funny,franchi...",3.5,3.213838,2004-09-10 03:07:30,Movie's title: Austin Powers: The Spy Who Shag...
1584,1379,Contact (1997),"Drama,Sci-Fi","adapted from:book,alien,aliens,astronauts,base...",3.5,3.629490,2004-09-10 03:07:36,"Movie's title: Contact (1997)\ngenres: Drama,S..."
1079,958,"Fish Called Wanda, A (1988)","Comedy,Crime","absurd,british,british comedy,caper,classic,cl...",4.0,3.813403,2004-09-10 03:07:45,"Movie's title: Fish Called Wanda, A (1988)\nge..."
...,...,...,...,...,...,...,...,...
5999,5149,Heavy Metal 2000 (2000),"Action,Adventure,Animation,Fantasy,Sci-Fi","animation,computer animation,immortality,origi...",3.5,3.094629,2005-04-02 23:55:50,Movie's title: Heavy Metal 2000 (2000)\ngenres...
7449,6176,Godsend (2004),"Drama,Horror,Thriller","alternate endings,bad script,clones,cloning,ge...",3.5,3.045714,2005-04-02 23:56:03,"Movie's title: Godsend (2004)\ngenres: Drama,H..."
4133,3631,Masters of the Universe (1987),"Action,Adventure,Fantasy,Sci-Fi","80s,action,based on a comic,based on a tv show...",3.0,2.805199,2005-04-02 23:56:09,Movie's title: Masters of the Universe (1987)\...


### Hàm recommend top k phim giống với một phim nhất

In [10]:
def get_top_k(movie, k=5):
    embed_input = index.reconstruct(movie["vectorID"])
    D, I = index.search(np.array([embed_input]), k)

    return D, I

### Tính điểm ranking của các phim được recommend từ mỗi phim mà user đã xem

In [10]:
recommend_dict = {}
negative_threshold = 3
negative_alpha = -0.5


for movieId, row in movie_list.iterrows():

    m_id = movieId

    # m_content = row["page_content"]

    m_rating = row["rating"]

    m_weight_rating = row["weight_rating"]

    combine_rating = 0.8 * m_rating + 0.2 * m_weight_rating

    D, I = get_top_k(row, k=5)

    rcm_ids = I.flatten().tolist()[1:]

    rcm_dists = D.flatten().tolist()[1:]

    for id_, dist_ in zip(rcm_ids, rcm_dists):
        if (m_rating < negative_threshold):
            wgt = (combine_rating + negative_alpha *
                   (negative_threshold - m_rating)) * dist_
        else:

            wgt = combine_rating * dist_

        recommend_dict[id_] = recommend_dict.get(id_, 0) + wgt

### Lấy ra top K recommend cuối cùng

In [11]:
K = 10
sorted_candidates = sorted(recommend_dict.items(),
                           key=lambda x: x[1], reverse=True)
final_rcm_list = [id_ for id_, _ in sorted_candidates[:K]]
final_rcm_list

[968, 1124, 1061, 3607, 1059, 4440, 2710, 1382, 1052, 1065]

In [13]:
movie_df.iloc[final_rcm_list]

,vectorID,title,genres,tags,weight_rating,imdbId,page_content
movieId,,,,,,,
1089,968,Reservoir Dogs (1992),"Crime,Mystery,Thriller","blood,bloody,brutal,brutality,caper,classic,cl...",4.052335,tt0105236,Movie's title: Reservoir Dogs (1992)\ngenres: ...
1275,1124,Highlander (1986),"Action,Adventure,Fantasy","1980s,action,adventure,alternate reality,aweso...",3.579709,tt0091203,Movie's title: Highlander (1986)\ngenres: Acti...
1210,1061,Star Wars: Episode VI - Return of the Jedi (1983),"Action,Adventure,Sci-Fi","action,action packed,adventure,alien,aliens,an...",3.984655,tt0086190,Movie's title: Star Wars: Episode VI - Return ...
4105,3607,"Evil Dead, The (1981)","Fantasy,Horror,Thriller","atmospheric,blood,bloody,brutality,classic,cla...",3.603036,tt0083907,"Movie's title: Evil Dead, The (1981)\ngenres: ..."
1208,1059,Apocalypse Now (1979),"Action,Drama,War","70mm,adapted from:book,amazing cinematography,...",4.063933,tt0078788,Movie's title: Apocalypse Now (1979)\ngenres: ...
5039,4440,Dragonslayer (1981),"Action,Adventure,Fantasy","adventure,cgi,computer animation,dark fantasy,...",3.284699,tt0082288,Movie's title: Dragonslayer (1981)\ngenres: Ac...
3070,2710,Adventures of Buckaroo Banzai Across the 8th D...,"Adventure,Comedy,Sci-Fi","80s,absurd,alien,aliens,campy,cult,cult classi...",3.341877,tt0086856,Movie's title: Adventures of Buckaroo Banzai A...
1587,1382,Conan the Barbarian (1982),"Action,Adventure,Fantasy","80s,action,adventure,arnold,awesome soundtrack...",3.224487,tt0082198,Movie's title: Conan the Barbarian (1982)\ngen...
1200,1052,Aliens (1986),"Action,Adventure,Horror,Sci-Fi","action,action packed,alien,alien invasion,alie...",3.971877,tt0090605,"Movie's title: Aliens (1986)\ngenres: Action,A..."


### LLM để phân phim thành nhiều thể loại

In [14]:
recommend_contents = "\n======\n".join(
    movie_df.iloc[final_rcm_list]["page_content"].tolist())

In [15]:
system_prompt = (
    "You are an AI assistant majoring in categorize things. Your task is to group the given movies by their genres into categories"
    """Your task is as follows:
        1. Group the movies into categories based on their genres.
        2. If a movie has multiple genres, assign it to the category corresponding to its primary genre (choose the first listed genre).
        3. For each category, list all movies that belong to that category."""
)

user_prompt = f"""
Below is a list of movies. Each movie is formatted as follows:

--------------------------------------------------
Movie Format:
(Title)
(Genres)        // A comma-separated list of genres.
(Tags)          // A comma-separated list of tags.
--------------------------------------------------
For example:
Now, Voyager (1942)
Drama, Romance
Classic, Timeless, Iconic
--------------------------------------------------

Please group these movies into categories based on their genres in the following format:
**Genre1**
- Movie1
- Movie2
...

**Genre2**
- Movie1
...

Here are the movies:
{recommend_contents}
"""

In [16]:
res = client.models.generate_content(
    model=GEN_MODEL,
    contents=[system_prompt, user_prompt]
)

In [17]:
print(res.text)

Here are the movie groupings by genre:

**Crime**
- Reservoir Dogs (1992)

**Action**
- Highlander (1986)
- Star Wars: Episode VI - Return of the Jedi (1983)
- Apocalypse Now (1979)
- Dragonslayer (1981)
- Conan the Barbarian (1982)
- Aliens (1986)

**Adventure**
- Adventures of Buckaroo Banzai Across the 8th Dimension, The (1984)

**Fantasy**
- Evil Dead, The (1981)

**Horror**
- Alien (1979)



In [ ]:
system_prompt_2 = "You are a JSON formatter AI assistant, your job is convert given data into valid JSON format."
user_prompt_2 = f"""Please convert the following text into JSON format as follow:  
{{
  "Genre1": [
    {{"title": "Movie Title 1"}},
    {{"title": "Movie Title 3"}}
  ],
  "Genre2": [
    {{"title": "Movie Title 1"}},
    {{"title": "Movie Title 5"}},
    ...
  ],
  ...
}}


Here's the text:
{res.text}

Please return ONLY the JSON in the format above, WITHOUT any explaination or infos.

"""

In [19]:
res2 = client.models.generate_content(
    model=GEN_MODEL,
    contents=[system_prompt_2, user_prompt_2]
)

In [20]:
print(res2.text)

```json
{
  "Crime": [
    {
      "title": "Reservoir Dogs (1992)"
    }
  ],
  "Action": [
    {
      "title": "Highlander (1986)"
    },
    {
      "title": "Star Wars: Episode VI - Return of the Jedi (1983)"
    },
    {
      "title": "Apocalypse Now (1979)"
    },
    {
      "title": "Dragonslayer (1981)"
    },
    {
      "title": "Conan the Barbarian (1982)"
    },
    {
      "title": "Aliens (1986)"
    }
  ],
  "Adventure": [
    {
      "title": "Adventures of Buckaroo Banzai Across the 8th Dimension, The (1984)"
    }
  ],
  "Fantasy": [
    {
      "title": "Evil Dead, The (1981)"
    }
  ],
  "Horror": [
    {
      "title": "Alien (1979)"
    }
  ]
}
```


---

# Đánh giá

In [7]:
from sklearn.model_selection import train_test_split
import time

---

### Cách 1: Sử dụng LLM để embed thông tin của phim (Content based)

In [ ]:
def get_rcm_dict(movies, k=5, negative_alpha=-0.5, negative_threshold=3):
    user_dislike = []
    user_like = []

    recommend_dict = {}

    for movieId, row in movies.iterrows():

        m_id = movieId

        m_content = row["page_content"]


        m_rating = row["rating"]

        m_weight_rating = row["weight_rating"]


        combine_rating = 0.8 * m_rating + 0.2 * m_weight_rating


        D, I = get_top_k(row, k=k)


        rcm_ids = I.flatten().tolist()[1:]

        rcm_dists = D.flatten().tolist()[1:]


        for id_, dist_ in zip(rcm_ids, rcm_dists):


            if (m_rating < negative_threshold):
                if (id_ in user_like):
                    continue
                user_dislike.append(row["vectorID"])

                wgt = (combine_rating + negative_alpha * (5 - m_rating)) * dist_

                recommend_dict[id_] = recommend_dict.get(id_, 0) - wgt


            else:
                user_like.append(row["vectorID"])

                wgt = combine_rating * dist_


                recommend_dict[id_] = recommend_dict.get(id_, 0) + wgt


        for id_ in set(user_dislike):
            recommend_dict.pop(id_, -1)

    return recommend_dict

In [13]:
user_size = user_rates.index.get_level_values(0).unique().shape[0]
K = [10, 20, 50]


metrics_results = {
    "HR": {k: [] for k in K},
    "Precision": {k: [] for k in K},
    "Recall": {k: [] for k in K},
    "NDCG": {k: [] for k in K},
}


counter = 0


for i in tqdm(range(1, user_size + 1)):

    precision = []
    recall_list = []
    ndcg_list = []

    if (counter == 5000):
        break
    user_ = user_rates.loc[i]

    if (user_.shape[0] < 100):
        continue

    if (counter % 100 == 0):
        print(counter)

    counter += 1

    user_ = user_.merge(movie_df, on="movieId")[
        ["vectorID", "title", "genres", "tags", "rating", "weight_rating", "timestamp", "page_content"]]

    train, test = train_test_split(user_, test_size=0.25, shuffle=False)

    recommend_dict = get_rcm_dict(train, k=10)

    for k_ in K:
        sorted_candidates = sorted(recommend_dict.items(),
                                   key=lambda x: x[1], reverse=True)

        final_rcm_list = movie_df.iloc[[id_ for id_,
                                        _ in sorted_candidates[:k_]]]

        overlap = np.intersect1d(final_rcm_list.index, test.index.values).size

        hr_k = 1 if overlap > 0 else 0

        precisionk = (overlap / k_)
        precision.append(precisionk)

        recall_k = overlap / len(test.index.values)
        recall_list.append(recall_k)

        test_movies = test.index.values

        relevances = [
            1 if movie in test_movies else 0 for movie in final_rcm_list]

        dcg = np.sum([rel / np.log2(idx + 2)
                      for idx, rel in enumerate(relevances)])

        ideal_hits = min(len(test_movies), k_)

        idcg = np.sum([1 / np.log2(i + 2) for i in range(ideal_hits)])

        ndcg_k = dcg / idcg if idcg > 0 else 0
        ndcg_list.append(ndcg_k)

        metrics_results["Precision"][k_].append(precisionk)

        metrics_results["Recall"][k_].append(recall_k)

        metrics_results["NDCG"][k_].append(ndcg_k)

        metrics_results["HR"][k_].append(hr_k)

  0%|          | 0/138493 [00:00<?, ?it/s]

0


  0%|          | 306/138493 [00:49<6:28:05,  5.93it/s] 

100


  0%|          | 545/138493 [01:38<4:22:53,  8.75it/s] 

...

  9%|▉         | 12437/138493 [40:01<5:08:03,  6.82it/s] 

4800


  9%|▉         | 12688/138493 [40:47<5:43:44,  6.10it/s] 

4900


  9%|▉         | 12989/138493 [41:40<6:42:43,  5.19it/s] 


In [14]:
final_results = {metric: [np.mean(metrics_results[metric][k])
                          for k in K] for metric in metrics_results}
results_df = pd.DataFrame(final_results, index=K).T

results_df

,10,20,50
HR,0.411000,0.637800,0.895800
Precision,0.059600,0.059480,0.059656
Recall,0.011657,0.023274,0.056090
NDCG,0.000000,0.000000,0.000000


In [16]:


final_results = {metric: [np.mean(metrics_results[metric][k])
                          for k in K] for metric in metrics_results}
results_df = pd.DataFrame(final_results, index=K).T

results_df

,10,20,50
HR,0.373913,0.547826,0.869565
Precision,0.050435,0.048696,0.049913
Recall,0.010261,0.019769,0.049409
NDCG,0.000000,0.000000,0.000000


---

### Cách 2: Sử dụng LLM để tạo ra vector phim đặc trưng của user

In [68]:
movie_list = user_rates.loc[2].merge(movie_df, on="movieId")[
    ["vectorID", "title", "genres", "tags", "rating", "weight_rating", "timestamp", "page_content"]]


def convert_prompt_2(row):
    rating = row["rating"]
    # link = f"https://www.imdb.com/title/{row["imdbId"]}/"
    # rating = round(row["weight_rating"],4)

    return f"{row["page_content"]} | rating: {rating}"


movie_list["page_content"] = movie_list.apply(convert_prompt_2, axis=1)

In [ ]:
def get_user_desc(contents):
    sys_prompt = """
    You are a recommendation system assistant. Your task is to analyze a user's watched movies and generate a concise feature vector representing their preferences. The feature vector must include relevant genres and descriptive tags.

    Extract common genres across the watched movies.
    Identify key descriptive tags based on themes, moods, and characteristics.
    If a movie has rating < 3.0 that means that user do not like that movie
    Format the output strictly as: 'genres: ... | tags: ...'.
    Do not include explanations, extra text, or formatting variations.
"""

    user_prompt = f"""
    Extract a feature vector from the user's watched movies. Use only genres and tags in the output. Format: 'genres: ... | tags: ...'. No extra text & strip all the unneccessary space.
    Here's the user watched movies:
    {contents}
"""

    res = client.models.generate_content(
        model=GEN_MODEL,
        contents=[sys_prompt, user_prompt]
    )

    return res


def _get_embed(feature_movies):
    res = client.models.embed_content(
        model=EMBED_MODEL,
        contents=feature_movies,
        config={
            "output_dimensionality": OUTPUT_DIMENSION
        }
    )

    embeddings = np.array([vector.values for vector in res.embeddings])
    normalized_vectors = embeddings / \
        np.linalg.norm(embeddings, axis=1, keepdims=True)

    return normalized_vectors


def get_top_k_2(feature_movie, k=10):
    return index.search(_get_embed(feature_movie), k)


def get_rcm(movies, k=10):
    recommend_contents = "\n======\n".join(
        movies["page_content"].tolist())

    user_desc = get_user_desc(recommend_contents)
    D, I = get_top_k_2(user_desc, k)

    return movie_df.iloc[I.flatten()]


def get_prefs(user_, K=50, threshold=3):
    high_rated = user_[user_["rating"] >= threshold]
    low_rated = user_[user_["rating"] < threshold]
    return user_.loc[high_rated.index.tolist()[:K//2] + low_rated.index.tolist()[:K//2]]

In [92]:
user_rates

rating           timestamp
userId movieId                            
1      924         3.5 2004-09-10 03:06:38
       919         3.5 2004-09-10 03:07:01
       2683        3.5 2004-09-10 03:07:30
       1584        3.5 2004-09-10 03:07:36
       1079        4.0 2004-09-10 03:07:45
...                ...                 ...
138493 6534        3.0 2009-12-07 18:18:28
       53464       4.0 2009-12-07 18:18:40
       1275        3.0 2010-01-01 20:42:32
       6996        3.0 2010-01-01 20:42:35
       405         3.0 2010-01-01 20:42:52

[19792548 rows x 2 columns]

In [112]:
descs = []
for i in tqdm(range(1, 201)):

    user_ = user_rates.loc[i]
    user_ = user_.merge(movie_df, on="movieId")[
        ["vectorID", "title", "genres", "tags", "rating", "weight_rating", "timestamp", "page_content"]]
    user_["page_content"] = user_.apply(convert_prompt_2, axis=1)
    train, test = train_test_split(user_, test_size=0.25, shuffle=False)
    user_prefs = get_prefs(train)

    big_str = "\n======\n".join(
        user_prefs["page_content"].tolist())
    user_desc = get_user_desc(big_str)
    time.sleep(3)

    descs.append(user_desc)

 57%|█████▊    | 115/200 [08:36<06:21,  4.49s/it]


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The service is currently unavailable.', 'status': 'UNAVAILABLE'}}

In [115]:
len(descs)

115

In [ ]:
import pickle

with open("data/1800-descriptions.pkl", "wb") as f:
    pickle.dump(descs, f)

In [ ]:
index_2 = faiss.IndexFlatIP(int(OUTPUT_DIMENSION))
index_2.add()

In [118]:
index_2.ntotal

115

In [133]:
batch_size = 100
batch_count = 0
# user_size = user_rates.index.get_level_values(0).unique().shape[0]
user_size = len(descs)
num_batches = (user_size + batch_size - 1) // batch_size

index_2 = faiss.IndexFlatIP(int(OUTPUT_DIMENSION))

for i in tqdm(range(0, user_size, batch_size)):
    temp = [i.text for i in descs]
    batch = temp[i:i+batch_size]
    embed_vectors = _get_embed(batch)
    batch_count += 1
    if (batch_count % 100 == 0):
        print(f"Processed batch {batch_count}/{num_batches}")
    if batch_count % 1500 == 0:
        print("Rate limit reached. Waiting for delay...")
        time.sleep(60)
    index_2.add(embed_vectors)

100%|██████████| 2/2 [00:03<00:00,  1.73s/it]


In [134]:
faiss.write_index(index_2, "../output_index/user_features_115")

In [140]:
descs[0].text

'genres: Adventure,Action,Comedy,Drama,Sci-Fi,Fantasy,Crime,Thriller,Horror,Romance,Western,Animation,Children,Mystery | tags: action,adventure,atmospheric,classic,comedy,cerebral,cinematography,cult classic,dark,drama,fantasy,funny,good,great,horror,humor,imdb top 250,masterpiece,mentor,original,philosophical,psychological,sci fi,scifi,science fiction,scary,storytelling,suspense,suspenseful,thriller,visually stunning,witty\n'

In [141]:
D, I = index.search(np.array([index_2.reconstruct(0)]), 5)

movie_df.iloc[I.flatten()]

,vectorID,title,genres,tags,weight_rating,imdbId,page_content
movieId,,,,,,,
1209,1060,Once Upon a Time in the West (C'era una volta ...,"Action,Drama,Western","amazing cinematography,atmospheric,beautifully...",3.931894,tt0064116,Movie's title: Once Upon a Time in the West (C...
1254,1103,"Treasure of the Sierra Madre, The (1948)","Action,Adventure,Drama,Western","adventure,afi 100,afi 100 (movie quotes),betra...",3.980917,tt0040897,"Movie's title: Treasure of the Sierra Madre, T..."
7099,5946,Nausicaä of the Valley of the Wind (Kaze no ta...,"Adventure,Animation,Drama,Fantasy,Sci-Fi","adventure,allegory,animated,animation,anime,ch...",3.860125,tt0087544,Movie's title: Nausicaä of the Valley of the W...
3000,2646,Princess Mononoke (Mononoke-hime) (1997),"Action,Adventure,Animation,Drama,Fantasy","adventure,animal movie,animated,animation,anim...",3.996854,tt0119698,Movie's title: Princess Mononoke (Mononoke-him...
2571,2255,"Matrix, The (1999)","Action,Sci-Fi,Thriller","action,action packed,adventure,allegory,alone ...",4.164415,tt0133093,"Movie's title: Matrix, The (1999)\ngenres: Act..."


In [137]:
# user_size = user_rates.index.get_level_values(0).unique().shape[0]
user_size = index_2.ntotal
K = [10, 20, 50]

metrics_results_2 = {
    "HR": {k: [] for k in K},
    "Precision": {k: [] for k in K},
    "Recall": {k: [] for k in K},
    "NDCG": {k: [] for k in K},
}

counter = 0

for i in tqdm(range(1, user_size+1)):
    precision = []
    recall_list = []
    ndcg_list = []

    if (counter == 1000):
        break

    user_ = user_rates.loc[i]

    counter += 1
    if (counter % 100 == 0):
        print(f"User count: {counter}")

    user_ = user_.merge(movie_df, on="movieId")[
        ["vectorID", "title", "genres", "tags", "rating", "weight_rating", "timestamp", "page_content"]]

    # print(user_)

    train, test = train_test_split(user_, test_size=0.25, shuffle=False)

    user_description_embed = np.array([index_2.reconstruct(i-1)])

    for k_ in K:
        D, I = index.search(user_description_embed, k_)

        recommend_movies = movie_df.iloc[I.flatten()]

        final_rcm_list = recommend_movies
        overlap = np.intersect1d(final_rcm_list.index, test.index.values).size
        hr_k = 1 if overlap > 0 else 0

        precisionk = (overlap / k_)
        precision.append(precisionk)

        recall_k = overlap / len(test.index.values)
        recall_list.append(recall_k)

        test_movies = test.index.values
        relevances = [
            1 if movie in test_movies else 0 for movie in final_rcm_list]
        dcg = np.sum([rel / np.log2(idx + 2)
                      for idx, rel in enumerate(relevances)])

        ideal_hits = min(len(test_movies), k_)
        idcg = np.sum([1 / np.log2(i + 2) for i in range(ideal_hits)])

        ndcg_k = dcg / idcg if idcg > 0 else 0
        ndcg_list.append(ndcg_k)

        metrics_results_2["Precision"][k_].append(precisionk)
        metrics_results_2["Recall"][k_].append(recall_k)
        metrics_results_2["NDCG"][k_].append(ndcg_k)
        metrics_results_2["HR"][k_].append(hr_k)

100%|██████████| 115/115 [00:01<00:00, 87.49it/s]

User count: 100


In [138]:
final_results_2 = {metric: [np.mean(metrics_results_2[metric][k])
                            for k in K] for metric in metrics_results_2}
results_df_2 = pd.DataFrame(final_results_2, index=K).T

results_df_2

,10,20,50
HR,0.095652,0.165217,0.356522
Precision,0.011304,0.011739,0.012522
Recall,0.007084,0.014885,0.033748
NDCG,0.000000,0.000000,0.000000


In [44]:
high_rated = movie_list[movie_list["rating"] >= 3.0]
low_rated = movie_list[movie_list["rating"] < 3.0]

In [52]:
k = 50

In [55]:
temp = high_rated.index.tolist()[:k//2] + low_rated.index.tolist()[:k//2]

In [ ]:
res = get_rcm(movie_list.loc[temp])

In [88]:
res.loc[541]["page_content"]

"Movie's title: Blade Runner (1982)\ngenres: Action,Sci-Fi,Thriller\ntags: adapted from:book,allegory,amazing cinematography,android(s)/cyborg(s),androids,artificial intelligence,atmospheric,based on a book,based on book,bleak,cerebral,cinematography,classic,clones,cloning,complex,criterion,cult classic,cult film,cyberpunk,cyborgs,dark,dark hero,dialogue,distopia,dystopia,dystopic future,enigmatic,excellent script,existentialism,fighting the system,film noir,future,futuristic,genetics,good soundtrack,great cinematography,great ending,grim,humanity,imdb top 250,intelligent sci-fi,interesting,loneliness,los angeles,man versus machine,masterpiece,melancholic,melancholy,memory,mindfuck,narrated,neo-noir,nocturnal,noir,noir thriller,ominous,original,original plot,philip k. dick,philosophical,philosophy,post apocalyptic,quotable,reflective,robots,sci fi,sci-fi,science fiction,scifi,slow paced,special effects,story,storytelling,stunning,stylized,technology,thought-provoking,violent,visual,vis